# 掼蛋·扑克牌识别模型训练（Colab 一键版）

在 **Google Colab 免费 GPU** 上训练识别扑克牌「数字+花色」的模型(YOLO)。

**用法**：菜单「运行时 → 更改运行时类型 → 选 GPU」，再「运行时 → 全部运行」。


## 1. 确认已分到 GPU


In [ ]:
!nvidia-smi

## 2. 安装训练框架（Ultralytics YOLO）


In [ ]:
!pip -q install ultralytics roboflow
import ultralytics; ultralytics.checks()

## 3. 获取扑克牌数据集（自动挑可用版本）

把 `api_key` 换成你 Roboflow 的 Private API Key；代码会自动列出版本、逐个尝试，找到带 YOLOv8 导出的就用。


In [ ]:
import shutil, os, glob, zipfile
from roboflow import Roboflow

rf = Roboflow(api_key="把你的Private_API_Key粘到这里")
project = rf.workspace("augmented-startups").project("playing-cards-ow27d")

try:
    vers = [v.version for v in project.versions()]
except Exception:
    vers = [1, 2, 3, 4, 5, 6]
print('要尝试的版本:', vers)

for d in glob.glob('/content/Playing-Cards-*') + glob.glob('/content/playing-cards-*'):
    shutil.rmtree(d, ignore_errors=True)

dataset = None
for vn in vers:
    try:
        ds = project.version(vn).download("yolov8")
    except Exception as e:
        print(f'版本{vn}: 跳过（{type(e).__name__}）'); continue
    loc = ds.location
    if not glob.glob(os.path.join(loc, '**/data.yaml'), recursive=True):
        for z in glob.glob(os.path.join(loc, '*.zip')):
            try:
                with zipfile.ZipFile(z) as zf: zf.extractall(loc)
            except Exception: pass
    if glob.glob(os.path.join(loc, '**/data.yaml'), recursive=True):
        dataset = ds; print(f'✅ 版本{vn} 可用 -> {loc}'); break
    print(f'版本{vn}: 无 YOLOv8 导出，换下一个')

assert dataset, '所有版本都没有 YOLOv8 导出——需在数据集网页 Generate 一个 YOLOv8 版本'
print('最终目录内容:', os.listdir(dataset.location))

## 4. 开始训练（20轮 + 一半数据，约1小时跑完）


In [ ]:
import glob, os
from ultralytics import YOLO
DATA = glob.glob(os.path.join(dataset.location, '**/data.yaml'), recursive=True)[0]
print('使用数据集配置:', DATA)
model = YOLO('yolov8n.pt')
model.train(data=DATA, epochs=20, imgsz=640, batch=16, fraction=0.5,
            patience=10, project='guandan_cards', name='exp')

## 5. 看效果（验证集指标）


In [ ]:
metrics = model.val()
print('mAP50:', metrics.box.map50, ' mAP50-95:', metrics.box.map)

## 6. 导出模型 + 下载（自动找 best.pt，不怕 exp-N 改名）


In [ ]:
import glob, os
from ultralytics import YOLO
from google.colab import files

cands = sorted(glob.glob('/content/**/weights/best.pt', recursive=True), key=os.path.getmtime)
assert cands, '没找到 best.pt'
best = cands[-1]
print('找到模型:', best)
m = YOLO(best)
onnx_path = m.export(format='onnx')
print('ONNX 路径:', onnx_path)
files.download(best)
files.download(onnx_path)

## 7.（已并入第6格）拿回模型后怎么用

把 `best.onnx` 上传到本仓库 `webapp/`，开 GitHub Pages，手机打开 `webapp/camera.html` 即可摄像头认牌。
或把 `best.pt` 放到 `vision/`，`python vision/recognize.py 照片.jpg` 跑识别+建议。
